# **Generating Higher Ed Text Data**



## **1. Importing the Course 2 Data**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

import plotly.express as px

pd.options.display.max_columns = None

In [ ]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 2 v2/data/'
df_training = pd.read_csv(f'{filepath}training.csv')
display(df_training)

,SID,COHORT,RACE_ETHNICITY,GENDER,FIRST_GEN_STATUS,HS_GPA,HS_MATH_GPA,HS_ENGL_GPA,COLLEGE,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2,UNITS_ATTEMPTED_3,UNITS_COMPLETED_1,UNITS_COMPLETED_2,UNITS_COMPLETED_3,DFW_UNITS_1,DFW_UNITS_2,DFW_UNITS_3,GPA_1,GPA_2,GPA_3,SEM_3_STATUS,DFW_RATE_1,DFW_RATE_2,GRADE_POINTS_1,GRADE_POINTS_2,GRADE_POINTS_3
0,UHDOP5522,Fall 2020,Asian,Female,Continuing Generation,3.720,3.2,3.400,Visual & Performing Arts,15.0,14.0,15.0,15.0,15.0,16.0,0.0,0.0,0.0,4.000000,3.785714,4.000000,E,0.0000,0.000000,60.0,53.0,60.0
1,UHE842CU6,Fall 2021,Black or African American,Female,Continuing Generation,3.189,2.6,3.750,Visual & Performing Arts,12.0,12.0,12.0,12.0,12.0,6.0,3.0,4.0,12.0,3.000000,2.500000,1.500000,E,0.0000,0.000000,36.0,30.0,18.0
2,UHJFT1JAB,Fall 2018,Asian,Female,Continuing Generation,3.625,3.4,3.500,Visual & Performing Arts,15.0,15.0,15.0,15.0,16.0,16.0,0.0,0.0,0.0,3.800000,3.600000,3.600000,E,0.0000,0.000000,57.0,54.0,54.0
3,UHKF05TAF,Fall 2018,Hispanic,Female,First Generation,3.606,3.0,3.375,Letters & Humanities,16.0,9.0,12.0,7.0,3.0,12.0,9.0,9.0,3.0,1.562500,1.000000,2.500000,E,0.5625,0.666667,25.0,9.0,30.0
4,UHKKQ8UY5,Fall 2021,Hispanic,Male,Continuing Generation,3.536,2.5,2.625,Letters & Humanities,13.0,13.0,15.0,13.0,13.0,15.0,0.0,0.0,0.0,3.538462,3.769231,3.400000,E,0.0000,0.000000,46.0,49.0,51.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19839,N2HOWBXJM,Fall 2019,Hispanic,Female,First Generation,3.467,3.4,3.833,Natural and Mathematical Sciences,10.0,13.0,14.0,10.0,13.0,14.0,0.0,0.0,0.0,3.000000,3.384615,2.428571,E,0.0000,0.000000,30.0,44.0,34.0
19840,N2JGRD0V6,Fall 2018,White,Female,Continuing Generation,3.956,3.5,4.333,General Studies,15.0,15.0,15.0,15.0,16.0,15.0,0.0,0.0,0.0,3.200000,4.000000,3.000000,E,0.0000,0.000000,48.0,60.0,45.0
19841,N2K6479P0,Fall 2018,Two or More Races,Female,Continuing Generation,3.912,3.0,4.000,General Studies,15.0,16.0,15.0,15.0,16.0,15.0,0.0,0.0,0.0,3.400000,3.437500,3.600000,E,0.0000,0.000000,51.0,55.0,54.0
19842,N2LXOSYTW,Fall 2018,Asian,Male,Continuing Generation,2.656,2.5,3.600,Letters & Humanities,12.0,13.0,11.0,3.0,7.0,11.0,12.0,6.0,7.0,1.000000,1.538462,2.818182,E,0.7500,0.461538,12.0,20.0,31.0


In [ ]:
pd.value_counts(df_training['COHORT'])

/tmp/ipython-input-3141611063.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(df_training['COHORT'])


,count
COHORT,
Fall 2019,5162
Fall 2018,4925
Fall 2020,4901
Fall 2021,4856


As proof of concept, we'll work with the F19 cohort.

In [ ]:
df_training19 = df_training[df_training['COHORT']=="Fall 2019"]
df_training19

,SID,COHORT,RACE_ETHNICITY,GENDER,FIRST_GEN_STATUS,HS_GPA,HS_MATH_GPA,HS_ENGL_GPA,COLLEGE,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2,UNITS_ATTEMPTED_3,UNITS_COMPLETED_1,UNITS_COMPLETED_2,UNITS_COMPLETED_3,DFW_UNITS_1,DFW_UNITS_2,DFW_UNITS_3,GPA_1,GPA_2,GPA_3,SEM_3_STATUS,DFW_RATE_1,DFW_RATE_2,GRADE_POINTS_1,GRADE_POINTS_2,GRADE_POINTS_3
5,UHKR1SFYP,Fall 2019,Asian,Male,Continuing Generation,3.500,2.667,4.167,Letters & Humanities,16.0,15.0,9.0,16.0,15.0,12.0,0.0,0.0,0.0,3.437500,3.800000,3.000000,E,0.00,0.000000,55.0,57.0,27.0
8,UHNF02H9O,Fall 2019,Hispanic,Female,First Generation,2.944,4.000,3.500,General Studies,15.0,13.0,12.0,15.0,13.0,15.0,0.0,0.0,0.0,2.600000,4.000000,3.750000,E,0.00,0.000000,39.0,52.0,45.0
9,UHOFN46WE,Fall 2019,Black or African American,Female,Continuing Generation,4.069,3.400,4.125,Business Administration,12.0,12.0,3.0,12.0,15.0,15.0,0.0,0.0,0.0,3.500000,4.000000,4.000000,E,0.00,0.000000,42.0,48.0,12.0
10,UHP7B9BTV,Fall 2019,Hispanic,Female,First Generation,3.032,1.250,2.625,Business Administration,12.0,9.0,12.0,12.0,6.0,0.0,3.0,6.0,12.0,1.500000,2.000000,0.750000,E,0.00,0.333333,18.0,18.0,9.0
11,UI30HVLIL,Fall 2019,Hispanic,Female,First Generation,3.273,3.250,2.800,Business Administration,15.0,13.0,16.0,15.0,13.0,13.0,0.0,3.0,3.0,3.200000,3.076923,2.625000,E,0.00,0.000000,48.0,40.0,42.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19832,N26B5Y6HE,Fall 2019,Nonresident alien,Female,First Generation,3.281,3.000,3.667,Letters & Humanities,16.0,9.0,12.0,16.0,9.0,15.0,0.0,0.0,0.0,2.937500,4.000000,3.250000,E,0.00,0.000000,47.0,36.0,39.0
19834,N28LHIN72,Fall 2019,White,Female,Continuing Generation,3.621,2.800,3.667,Natural and Mathematical Sciences,13.0,13.0,15.0,13.0,14.0,15.0,0.0,0.0,0.0,2.769231,3.769231,3.066667,E,0.00,0.000000,36.0,49.0,46.0
19835,N299IIGIN,Fall 2019,Two or More Races,Female,Continuing Generation,4.094,3.667,4.167,Visual & Performing Arts,15.0,15.0,12.0,15.0,16.0,13.0,0.0,0.0,0.0,4.000000,4.000000,4.000000,E,0.00,0.000000,60.0,60.0,48.0
19837,N2C5YLPI1,Fall 2019,Asian,Male,Continuing Generation,3.658,3.750,3.167,Engineering & Technology,16.0,16.0,16.0,12.0,3.0,16.0,4.0,13.0,0.0,2.125000,1.125000,3.500000,E,0.25,0.812500,34.0,18.0,56.0


We'll need to use HS GPA as a proxy for student perspective. Synthetic data will be generated to correlate with HS GPA.

In [ ]:
df_students_gpa = df_training19[['SID', 'HS_GPA']].copy()

# Check for missing values
missing_gpa_count = df_students_gpa['HS_GPA'].isnull().sum()
print(f"Number of missing HS_GPA values: {missing_gpa_count}")

# Fill missing values with the mean
if missing_gpa_count > 0:
    mean_gpa = df_students_gpa['HS_GPA'].mean()
    df_students_gpa['HS_GPA'].fillna(mean_gpa, inplace=True)
    print(f"Filled missing HS_GPA values with the mean: {mean_gpa:.2f}")

# Normalize HS_GPA
scaler = StandardScaler()
df_students_gpa['HS_GPA_normalized'] = scaler.fit_transform(df_students_gpa[['HS_GPA']])

# Display the first few rows of the updated DataFrame
print("\nDataFrame after processing:")
display(df_students_gpa.head())

Number of missing HS_GPA values: 0

DataFrame after processing:


,SID,HS_GPA,HS_GPA_normalized
5,UHKR1SFYP,3.500,-0.304547
8,UHNF02H9O,2.944,-1.824803
9,UHOFN46WE,4.069,1.251254
10,UHP7B9BTV,3.032,-1.584187
11,UI30HVLIL,3.273,-0.925227


## **2. Generating Synthetic Survey Data**

### **2.1 NSSE-like Ordinal Survey Questions**

Here are the 10 simulated NSSE-like survey questions, each with a 1-4 response scale:

**Response Scale:**
1 = Never / Strongly Disagree
2 = Sometimes / Disagree
3 = Often / Agree
4 = Very Often / Strongly Agree

**Survey Questions:**

1.  To what extent have you participated in collaborative learning activities?
2.  How often have you initiated discussion of course topics with faculty outside of class?
3.  How often have you worked with classmates on projects or assignments?
4.  To what extent has your coursework emphasized analytical and problem-solving skills?
5.  How often have you connected learning to societal problems or issues?
6.  How much has your institution emphasized providing support to help you succeed academically?
7.  To what extent have you gained a deeper understanding of people from different backgrounds?
8.  How often have you participated in co-curricular activities (e.g., clubs, organizations, events)?
9.  How often have you thought critically about your the requirements of your major?
10. To what extent has your academic experience contributed to your personal growth and development?

| Category | Survey Questions |
|----------|------------------|
| (1) Participation in educationally purposeful activities | - To what extent have you participated in collaborative learning activities?<br>- How often have you worked with classmates on projects or assignments?<br>- How often have you connected learning to societal problems or issues?|
| (2) Institutional requirements and the challenging nature of coursework | - To what extent has your coursework emphasized analytical and problem-solving skills?<br>- How often have you thought critically about your the requirements of your major?  |
| (3) Perceptions of the college environment | - How much has your institution emphasized providing support to help you succeed academically?<br>- How often have you participated in co-curricular activities (e.g., clubs, organizations, events)? |
| (4) Estimates of educational and personal growth | - To what extent have you gained a deeper understanding of people from different backgrounds?<br>- How often have you initiated discussion of course topics with faculty outside of class?<br>- To what extent has your academic experience contributed to your personal growth and development? |
| (5) Background and demographic information | - None of the listed questions fall into this category. |


In [ ]:
import numpy as np
import pandas as pd

# 1. Initialize an empty DataFrame called Survey_training19 and populate it with the 'SID' column
Survey_training19 = df_students_gpa[['SID']].copy()

# 2. Create a loop that iterates 10 times to generate responses for 10 survey questions.
num_questions = 10
for i in range(1, num_questions + 1):
    question_name = f'NSSE_Q{i}'

    # 3. Generate responses moderately correlated with 'HS_GPA_normalized'
    # Scale HS_GPA_normalized to roughly fit within the 1-4 range, add noise, and shift.
    # A coefficient for HS_GPA_normalized will determine the strength of correlation.
    # np.random.uniform(0, 1) generates noise between 0 and 1.
    # The base value (e.g., 2.5) helps center the responses around the middle of the 1-4 scale.
    # The noise magnitude (e.g., 1.5) adds variability.

    # Example of a moderate correlation: scale normalized GPA by a factor (e.g., 0.5 to 1.0)
    # and add uniform random noise. Adjust these parameters for desired correlation strength.
    correlation_strength = 0.7 # Adjust this value to change correlation strength
    noise_level = 1.0 # Adjust this for more or less randomness

    simulated_responses = (df_students_gpa['HS_GPA_normalized'] * correlation_strength) \
                          + (np.random.uniform(0, 1, size=len(df_students_gpa)) * noise_level) \
                          + 2.0 # Base value to center responses around the middle of the 1-4 scale

    # 4. Ensure that all generated responses are integers and fall within the 1-4 scale
    # Clip values between 1 and 4, then convert to integers.
    Survey_training19[question_name] = np.clip(np.round(simulated_responses), 1, 4).astype(int)

# 6. After the loop, display the first few rows of the Survey_training19 DataFrame
print("First few rows of Survey_training19:")
Survey_training19

First few rows of Survey_training19:


,SID,NSSE_Q1,NSSE_Q2,NSSE_Q3,NSSE_Q4,NSSE_Q5,NSSE_Q6,NSSE_Q7,NSSE_Q8,NSSE_Q9,NSSE_Q10
5,UHKR1SFYP,2,3,2,2,2,2,2,2,3,2
8,UHNF02H9O,2,2,1,1,1,2,1,1,1,1
9,UHOFN46WE,3,3,4,3,4,3,3,3,3,3
10,UHP7B9BTV,2,1,2,1,1,2,1,1,2,1
11,UI30HVLIL,2,2,2,2,2,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...
19832,N26B5Y6HE,2,2,2,1,2,2,2,2,2,2
19834,N28LHIN72,2,3,3,2,3,3,2,3,3,3
19835,N299IIGIN,4,4,4,4,4,4,3,4,4,3
19837,N2C5YLPI1,3,3,3,2,3,3,3,2,3,3


### **2.2 Generating Free-Form Text Responses**

Generate a free-form text response for each student, where the content is subtly correlated with their normalized `HS_GPA`.

**Reasoning**:
First, I'll initialize an empty DataFrame `Free_Response_training19` and populate it with the 'SID' column from `df_students_gpa`. Then, I will define a function that generates free-form text responses, subtly varying the content based on the `HS_GPA_normalized` value to reflect different student experiences or sentiments. Finally, I will apply this function to each student's `HS_GPA_normalized` and store the results in the `Free_Response_training19` DataFrame, displaying its head to verify the output.



In [ ]:
import random

# 1. Initialize an empty DataFrame called Free_Response_training19 and populate it with the 'SID' column
Free_Response_training19 = df_students_gpa[['SID']].copy()

import random

def generate_free_response(gpa_normalized):
    # Adjust thresholds based on the distribution of your normalized GPA
    if gpa_normalized > 1.0:  # High GPA (top ~16% for standard normal)
        responses = [
            # --- keep existing ---
            "I feel well-prepared for college-level work and am eager to explore advanced topics.",
            "My academic journey has been very rewarding, and I'm looking forward to new challenges.",
            "I am confident in my study habits and ability to succeed in my chosen field.",
            "Education has always been a priority, and I strive for excellence in everything I do.",

            # --- added: Performance & Feedback ---
            "I usually understand instructor feedback quickly and can apply it to improve my work.",
            "I tend to do well on exams and projects, and I appreciate feedback that pushes me to refine my thinking.",
            "Rubrics and comments are helpful for fine-tuning my work, especially on larger assignments.",

            # --- added: Academic Preparedness & Challenge / Course Rigor ---
            "Most courses feel appropriately challenging, and I enjoy when assignments require deeper analysis.",
            "I like classes that move at a faster pace and expect us to connect concepts across topics.",
            "I feel academically prepared, so I’m comfortable taking on rigorous coursework.",

            # --- added: Study Habits & Peer Support ---
            "I plan my weeks in advance, and that structure helps me stay ahead of deadlines.",
            "Study groups help, but I mostly use them to compare approaches after I’ve done my own work first.",
            "I regularly review notes and practice problems, which keeps me from falling behind.",

            # --- added: Campus & Class Experience / Resources ---
            "I’ve had a strong classroom experience overall, and I feel comfortable participating in discussions.",
            "Office hours have been especially useful for going beyond the basics and clarifying expectations.",
            "I’ve found campus resources easy to access when I need something specific."
        ]

    elif gpa_normalized > -0.5:  # Average GPA
        responses = [
            # --- keep existing ---
            "I am adapting to college life and finding my rhythm, learning new things every day.",
            "My experience so far has been a mix of excitement and challenge, but I'm managing.",
            "I'm working hard to keep up with my coursework and improve my skills.",
            "College is an interesting experience, and I'm discovering what I'm passionate about.",

            # --- added: Performance & Feedback ---
            "I do best when I get clear feedback on what to improve, especially on writing and projects.",
            "Sometimes feedback helps a lot, but other times I’m not sure how to apply it on the next assignment.",
            "I feel like my performance depends on the course—some classes click, others take more time to figure out.",

            # --- added: Study Habits & Peer Support ---
            "I’m still figuring out the best way to study, and talking with classmates helps me stay on track.",
            "Group work can be helpful, but it depends on how organized the team is.",
            "When I study consistently, I do fine—my challenge is keeping that routine during busy weeks.",

            # --- added: Academic Support & Course Rigor / Preparedness ---
            "The workload can feel heavy at times, but I can manage when I start assignments early.",
            "Some classes feel more challenging than I expected, and I’m learning how to meet those expectations.",
            "I feel prepared in some areas, but there are topics where I need more practice or review.",

            # --- added: Campus & Class Experience / Resources ---
            "Overall I’ve had a positive experience in my classes, but I sometimes hesitate to speak up.",
            "I’ve used tutoring or office hours a few times, and it usually helps when I make the time.",
            "I’m learning what campus resources exist, but I don’t always remember to use them until I’m stressed.",

            # --- added: Workload & Balance ---
            "Balancing school with work and personal responsibilities is doable, but it takes planning.",
            "When multiple deadlines hit at once, I have to prioritize carefully to keep up."
        ]

    else:  # Lower GPA
        responses = [
            # --- keep existing ---
            "I'm finding some aspects of college challenging and am seeking resources to improve.",
            "The transition to college has been tougher than I expected, but I'm determined to grow.",
            "I need to focus more on my studies and develop better learning strategies.",
            "I hope to find my academic footing soon and start seeing better results.",

            # --- added: Struggle & Need for Help / Support ---
            "I’ve been feeling overwhelmed, and I’m trying to figure out where to get help before I fall further behind.",
            "Sometimes I don’t understand what instructors expect, and I wish I had more guidance earlier in the course.",
            "I know support exists, but it can be hard to reach out when I’m already stressed.",

            # --- added: Workload & Balance ---
            "My workload feels hard to manage, especially when I’m juggling work, family, and classes.",
            "I often start assignments too late, and then I’m rushing to finish instead of learning the material.",
            "When deadlines pile up, I feel like I’m just trying to survive the week.",

            # --- added: Academic Preparedness & Challenge / Course Rigor ---
            "Some course material moves too fast for me, and I need more time to build the basics.",
            "I feel like I’m catching up on foundational skills while also trying to keep up with new content.",
            "The rigor is higher than I expected, and I’m still learning how to meet that standard.",

            # --- added: Study Habits & Peer Support ---
            "I’m trying to improve my study habits, but I’m not always sure what strategies work best.",
            "Working with classmates helps, but I sometimes feel embarrassed asking questions I think I should already know.",
            "I want to build a better routine for studying, attending office hours, and reviewing feedback.",

            # --- added: Performance & Feedback / Campus Experience ---
            "I sometimes get feedback that I don’t fully understand, and I need help turning it into concrete next steps.",
            "I’ve had moments where I felt lost in class, and that makes it harder to stay motivated.",
            "I’m trying to stay engaged, but when I struggle early, it affects my confidence for the rest of the term.",

            # --- added: Campus Support & Resources ---
            "I think tutoring or academic coaching could help me, but I need a clearer path for how to start.",
            "I want to use more campus resources, but scheduling and time constraints get in the way."
        ]

    return random.choice(responses)
# 3. Apply the function to generate free responses
Free_Response_training19['Free_Response_Text'] = df_students_gpa['HS_GPA_normalized'].apply(generate_free_response)

# 4. Display the first few rows of the Free_Response_training19 DataFrame
print("First few rows of Free_Response_training19:")
display(Free_Response_training19.head())

First few rows of Free_Response_training19:


,SID,Free_Response_Text
5,UHKR1SFYP,"When multiple deadlines hit at once, I have to..."
8,UHNF02H9O,"When deadlines pile up, I feel like I’m just t..."
9,UHOFN46WE,"My academic journey has been very rewarding, a..."
10,UHP7B9BTV,Sometimes I don’t understand what instructors ...
11,UI30HVLIL,"I’ve been feeling overwhelmed, and I’m trying ..."


## **3. Vectorizing Text Data**

Let's implement `TfidfVectorizer`. This involves importing the necessary class, instantiating it with specified parameters, applying it to the `Free_Response_Text` column, and converting the output to a DataFrame.



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Instantiate TfidfVectorizer with specified parameters
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    ngram_range=(1, 1)
)

# Apply TfidfVectorizer to the 'Free_Response_Text' column
tfidf_matrix = tfidf_vectorizer.fit_transform(Free_Response_training19['Free_Response_Text'])

# Get feature names (vocabulary)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

# Convert the sparse matrix to a pandas DataFrame
df_tfidf_vectorized = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_feature_names)

# Display the first few rows and columns of the TfidfVectorized DataFrame
print("First few rows of TfidfVectorized DataFrame:")
df_tfidf_vectorized.index = Survey_training19.index
df_tfidf_vectorized
print(f"\nShape of TfidfVectorized DataFrame: {df_tfidf_vectorized.shape}")

First few rows of TfidfVectorized DataFrame:

Shape of TfidfVectorized DataFrame: (5162, 259)


Merging to create the raw survey data:

In [ ]:
# Merge Survey_training19 and df_tfidf_pca on their index values (SID)
ML_Survey_Data19 = pd.merge(Survey_training19, Free_Response_training19['Free_Response_Text'], left_index=True, right_index=True)

# Display the first few rows of the merged DataFrame
print("First few rows of the merged DataFrame (Ordinal and Free Response):")
display(ML_Survey_Data19.head())

print(f"\nShape of the merged DataFrame: {ML_Survey_Data19.shape}")

First few rows of the merged DataFrame (Ordinal and Free Response):


,SID,NSSE_Q1,NSSE_Q2,NSSE_Q3,NSSE_Q4,NSSE_Q5,NSSE_Q6,NSSE_Q7,NSSE_Q8,NSSE_Q9,NSSE_Q10,Free_Response_Text
5,UHKR1SFYP,2,3,2,2,2,2,2,2,3,2,"When multiple deadlines hit at once, I have to..."
8,UHNF02H9O,2,2,1,1,1,2,1,1,1,1,"When deadlines pile up, I feel like I’m just t..."
9,UHOFN46WE,3,3,4,3,4,3,3,3,3,3,"My academic journey has been very rewarding, a..."
10,UHP7B9BTV,2,1,2,1,1,2,1,1,2,1,Sometimes I don’t understand what instructors ...
11,UI30HVLIL,2,2,2,2,2,2,2,2,2,2,"I’ve been feeling overwhelmed, and I’m trying ..."



Shape of the merged DataFrame: (5162, 12)


Merging to create the numerical matrices from the data

In [ ]:
# Merge Survey_training19 and df_tfidf_pca on their index values (SID)
df_merged_surv_features = pd.merge(Survey_training19, df_tfidf_vectorized, left_index=True, right_index=True)

# Display the first few rows of the merged DataFrame
print("First few rows of the merged DataFrame (Survey responses + PCA-transformed text features):")
display(df_merged_surv_features.head())

print(f"\nShape of the merged DataFrame: {df_merged_surv_features.shape}")

First few rows of the merged DataFrame (Survey responses + PCA-transformed text features):


,SID,NSSE_Q1,NSSE_Q2,NSSE_Q3,NSSE_Q4,NSSE_Q5,NSSE_Q6,NSSE_Q7,NSSE_Q8,NSSE_Q9,NSSE_Q10,ability,academic,academically,access,adapting,advance,advanced,affects,ahead,analysis,apply,appreciate,approaches,appropriately,areas,asking,aspects,assignment,assignments,attending,balancing,basics,best,better,build,busy,campus,carefully,catching,challenge,challenges,challenging,chosen,clarifying,class,classes,classmates,classroom,clear,clearer,click,coaching,college,comfortable,comments,compare,concepts,concrete,confidence,confident,connect,consistently,constraints,content,course,courses,coursework,day,deadlines,deeper,depends,determined,develop,discovering,discussions,doable,don,eager,earlier,early,easy,education,embarrassed,engaged,enjoy,especially,exams,excellence,excitement,exist,exists,expect,expectations,expected,experience,explore,fall,falling,family,far,fast,faster,feedback,feel,feeling,feels,felt,field,figure,figuring,finding,fine,finish,focus,footing,forward,foundational,fully,going,group,groups,grow,guidance,habits,hard,harder,heavy,help,helpful,helps,hesitate,higher,hit,hope,hours,improve,instead,instructor,instructors,interesting,journey,juggling,just,keeping,keeps,know,larger,late,learning,level,life,like,looking,lost,lot,make,makes,manage,managing,material,meet,mix,moments,motivated,moves,multiple,need,new,notes,office,organized,overall,overwhelmed,pace,participating,passionate,path,performance,personal,pile,plan,planning,positive,practice,prepared,prioritize,priority,problems,projects,pushes,questions,quickly,reach,refine,regularly,remember,require,resources,responsibilities,rest,results,review,reviewing,rewarding,rhythm,rigor,rigorous,routine,rubrics,rushing,scheduling,school,seeing,seeking,skills,soon,speak,specific,standard,start,stay,steps,strategies,stressed,strive,strong,structure,struggle,studies,study,studying,succeed,support,sure,survive,takes,taking,talking,team,tend,term,things,think,thinking,time,times,topics,tougher,track,transition,trying,tuning,turning,tutoring,understand,use,used,useful,usually,ve,want,way,week,weeks,wish,work,working,workload,writing
5,UHKR1SFYP,2,3,2,2,2,2,2,2,3,2,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.457811,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.402039,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.457811,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.457811,0.0,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.457811,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
8,UHNF02H9O,2,2,1,1,1,2,1,1,1,1,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.309518,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.20448,0.0000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.416571,0.0,0.0,0.0,0.0,


Shape of the merged DataFrame: (5162, 270)


Export the final result

In [ ]:
import os

# Define the base path for your Google Drive
drive_path = '/content/drive/MyDrive/'

# Define the target directory for exporting the merged features
export_dir = os.path.join(drive_path, 'IR ML Cert/MLCert Course 3/Course 3 Data')

# Create the directory if it doesn't exist
os.makedirs(export_dir, exist_ok=True)

# Define the full file path for the CSV
output_filename = 'ML_Survey_Data19.csv'
output_filepath = os.path.join(export_dir, output_filename)

# Export the DataFrame to CSV without the index (as SID is a column)
ML_Survey_Data19.to_csv(output_filepath, index=False)

print(f"DataFrame successfully exported to: {output_filepath}")

DataFrame successfully exported to: /content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/ML_Survey_Data19.csv
